# mine-tuning-model

## 필수 라이브러리 설치

In [18]:
!pip install --upgrade chromadb langchain langchain-community langchain-openai sentence-transformers transformers datasets gradio langgraph -q

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip freeze > /content/drive/MyDrive/Colab Notebooks/Github/mine-tuning-model/requirements.txt

## 허깅페이스에서 데이터 가져오기

In [19]:
from datasets import load_dataset

ds = load_dataset("lparkourer10/minecraft-wiki")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['url', 'question', 'answer'],
        num_rows: 87934
    })
})


In [20]:
# 샘플 3개 출력
for i in range(3):
    print(f"=== 샘플 {i+1} ===")
    print(f"URL   : {ds['train']['url'][i]}")
    print(f"Q     : {ds['train']['question'][i]}")
    print(f"A     : {ds['train']['answer'][i][:200]}")  # 앞 200자만
    print()

=== 샘플 1 ===
URL   : https://minecraft.wiki/w/Minecraft_(franchise)
Q     : What are the main differences between the original Minecraft gameplay and its current version in Windows 10 Edition?
A     : The main difference between the original Minecraft gameplay and its current version in Windows 10 Edition is the addition of new features and game modes, which have been implemented through Microsoft'

=== 샘플 2 ===
URL   : https://minecraft.wiki/w/Minecraft_(franchise)
Q     : Is the Creative mode in Minecraft really as easy to understand and use as it is portrayed in the summary?
A     : No, the creative mode in Minecraft is not entirely easy to understand and use as portrayed in the summary. In fact, some aspects of creative mode can be challenging for new players to grasp. For examp

=== 샘플 3 ===
URL   : https://minecraft.wiki/w/Minecraft_(franchise)
Q     : How does the addition of new biomes, mobs, and game modes in the recent update enhance the overall gameplay experience for player

# 데이터 임베딩


In [21]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print('[입베딩 성공]')
print(f"임베딩 차원: {embedding_model.get_sentence_embedding_dimension()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[입베딩 성공]
임베딩 차원: 384


/tmp/ipykernel_748/1680344745.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"임베딩 차원: {embedding_model.get_sentence_embedding_dimension()}")


## Chroma DB 구축

In [22]:
import chromadb
from tqdm import tqdm

# 기존 컬렉션 삭제 후 새로 생성
chroma_client = chromadb.Client()
chroma_client.delete_collection(name="minecraft_rag")
collection = chroma_client.create_collection(name="minecraft_rag")

# 전체 데이터
total_size = len(ds["train"]["answer"])
answers   = ds["train"]["answer"]
questions = ds["train"]["question"]
urls      = ds["train"]["url"]

print(f"총 데이터 수: {total_size}개")

# 테스용 코드 (sample data = 5000)
# batch_size = 500
# sample_size = 5000
# for i in range(0, sample_size, batch_size):
#     batch_answers = answers[i:i+batch_size]
#     embeddings = embedding_model.encode(batch_answers, show_progress_bar=True).tolist()
#     collection.add(
#         documents=batch_answers,
#         embeddings=embeddings,
#         metadatas=[{"question": q, "url": u} for q, u in zip(questions[i:i+batch_size], urls[i:i+batch_size])],
#         ids=[str(j) for j in range(i, i+len(batch_answers))]
#     )
#     print(f" {min(i+batch_size, sample_size)}/{sample_size} 완료")

# 전체 데이터 로드
batch_size = 500
for i in tqdm(range(0, total_size, batch_size), desc="벡터 DB 구축 중"):
    batch_answers   = answers[i:i+batch_size]
    batch_questions = questions[i:i+batch_size]
    batch_urls      = urls[i:i+batch_size]

    embeddings = embedding_model.encode(
        batch_answers,
        show_progress_bar=False
    ).tolist()

    collection.add(
        documents=batch_answers,
        embeddings=embeddings,
        metadatas=[{"question": q, "url": u} for q, u in zip(batch_questions, batch_urls)],
        ids=[str(j) for j in range(i, i+len(batch_answers))]
    )

print(f"\n 완료! 총 {collection.count()}개 저장됨")

ImportError: cannot import name '_ON_EMIT_RECURSION_COUNT_KEY' from 'opentelemetry.context' (/usr/local/lib/python3.12/dist-packages/opentelemetry/context/__init__.py)

## 검색 테스트

In [ ]:
def retrieve(query: str, top_k: int = 3):
    query_embedding = embedding_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )
    return results["documents"][0], results["metadatas"][0]

# 테스트
docs, metas = retrieve("How to find diamonds?")
for i, (doc, meta) in enumerate(zip(docs, metas)):
    print(f"=== 검색 결과 {i+1} ===")
    print(f"URL     : {meta['url']}")
    print(f"Q       : {meta['question']}")
    print(f"Answer  : {doc[:200]}")
    print()

=== 검색 결과 1 ===
URL     : https://minecraft.wiki/w/Bottle_o%27_Enchanting
Q       : How do players obtain resources such as diamonds and obsidian naturally in the wilderness?
Answer  : In Minecraft, players can naturally obtain diamonds and obsidian by mining them from deep within the game's underground structures, such as caves. Diamonds are typically found in areas with low light 

=== 검색 결과 2 ===
URL     : https://minecraft.wiki/w/Saddle
Q       : What is the process of obtaining certain materials such as diamonds and emeralds that are required for crafting items like bows and arrows?
Answer  : In Minecraft, obtaining certain materials such as diamonds and emeralds typically requires mining them from the ground. Diamonds can be found in deep stone layers, while emeralds are often hidden with

=== 검색 결과 3 ===
URL     : https://minecraft.wiki/w/Leaves
Q       : How do Diamonds affect gameplay or resource gathering in Minecraft?
Answer  : Diamonds are used to craft tools such as pickax

## LLM 로드

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

llm = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False
)

print("LLM 로드 완료")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM 로드 완료


## RAG 파이프라인 완성

In [ ]:
def rag_answer(query: str) -> str:
    # 1. 관련 문서 검색
    docs, metas = retrieve(query, top_k=3)
    context = "\n\n".join(docs)

    # 2. 프롬프트 구성
    prompt = f"""You are a helpful Minecraft expert assistant.
Use the following context to answer the question accurately.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {query}
Answer:"""

    # 3. LLM 답변 생성
    messages = [{"role": "user", "content": prompt}]
    result = llm(messages)
    answer = result[0]["generated_text"][-1]["content"]

    # 4. 출처 출력
    print(f"출처: {metas[0]['url']}\n")
    return answer


In [ ]:
# 테스트
response = rag_answer("How to find diamonds?")
print(response)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


출처: https://minecraft.wiki/w/Bottle_o%27_Enchanting

To find diamonds in Minecraft, you need to mine them from deep within the game's underground structures, such as caves. Diamonds are typically found in areas with low light levels, like caves or dark biomes. Additionally, you can explore abandoned mineshafts, which contain both diamonds and obsidian.


## Gradio 연결

In [ ]:
import gradio as gr

def chat(message, history):
    answer = rag_answer(message)
    return answer

demo = gr.ChatInterface(
    fn=chat,
    title="⛏️ Minecraft Guide LLM",
    description="Minecraft Wiki 기반 AI 가이드에게 무엇이든 물어보세요!",
    examples=[
        "How to find diamonds?",
        "How do I defeat the Ender Dragon?",
        "How to make a Nether portal?"
    ],
)

demo.launch(share=True)  # share=True 로 외부 접속 링크 생성

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b78f06ca2efb0604f5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
